
# Preparação dos dados de campo — Simon Solar (PVDAQ)

Este notebook prepara os dados medidos da usina Simon Solar para comparação com os resultados simulados no pvlib.

Selecionamos as principais variáveis operacionais, agregamos as medições originais de 5 minutos em médias horárias e salvamos um arquivo CSV reduzido para as análises de desempenho.

Os resultados são identificados pelo **centro de cada intervalo horário (HH:30)**, seguindo a convenção dos dados meteorológicos NSRDB utilizados na simulação.

O arquivo resultante será utilizado no Tutorial 3 para exploração dos dados, controle de qualidade (QA/QC) e validação do modelo.


In [3]:
import pandas as pd
from pathlib import Path

# ------------------------------------------
# 1. LOAD SELECTED FIELD MEASUREMENTS
# ------------------------------------------
input_file = Path("9069_subset.csv")
output_file = Path("9069_subset_hourly.csv")
output_file.parent.mkdir(parents=True, exist_ok=True)

# Keep relevant measurements, including all 40 inverter AC power channels
def keep_column(col):
    return (
        col == "utc_measured_on"
        or col.startswith("meter_")
        or col.startswith("poa_irradiance")
        or col.startswith("reference_cell_poa")
        or col in [
            "ghi_irradiance",
            "module_temperature_median",
            "ambient_temperature_median",
            "wind_speed_ms",
            "wind_direction_deg",
        ]
        or (col.startswith("inverter_") and "_ac_power_" in col)
    ) and "power_factor" not in col

df = pd.read_csv(input_file, usecols=keep_column)

# Parse timestamps while respecting UTC offsets
df["utc_measured_on"] = pd.to_datetime(
    df["utc_measured_on"], utc=True
)

df = df.set_index("utc_measured_on").sort_index()

# Convert to site-local time
df.index = df.index.tz_convert("America/New_York")

# Convert measurements to numeric
df = df.apply(pd.to_numeric, errors="coerce")

print("Original measurements:", len(df))
print("Typical interval:", df.index.to_series().diff().median())

# ------------------------------------------
# 2. AGGREGATE TO HOURLY
# ------------------------------------------


# Average measurements into hourly intervals
# Each hour represents [HH:00, HH+1:00)
hourly_df = df.resample(
    "1h",
    closed="left",
    label="left"
).mean()

# Label each hourly average at the center of its interval
hourly_df.index += pd.Timedelta(minutes=30)

# Track completeness of both AC meter measurements
for col in ["meter_1_ac_power_kw", "meter_2_ac_power_kw"]:
    if col in df.columns:
        hourly_df[col + "_n_samples"] = (
            df[col].resample("1h").count()
        )

# ------------------------------------------
# 3. SAVE FILE
# ------------------------------------------
hourly_df.to_csv(output_file, index_label="timestamp")

print(f"Saved: {output_file}")
print("Hourly records:", len(hourly_df))
print("Number of columns:", len(hourly_df.columns))

display(hourly_df.head())


Original measurements: 769980
Typical interval: 0 days 00:05:00
Saved: 9069_subset_hourly.csv
Hourly records: 68205
Number of columns: 54


,meter_1_ac_power_kw,meter_2_ac_power_kw,poa_irradiance_02a,poa_irradiance_02b,poa_irradiance_08,ghi_irradiance,reference_cell_poa_median,reference_cell_poa_1,module_temperature_median,ambient_temperature_median,...,inverter_33_ac_power_(kw),inverter_34_ac_power_(kw),inverter_35_ac_power_(kw),inverter_36_ac_power_(kw),inverter_37_ac_power_(kw),inverter_38_ac_power_(kw),inverter_39_ac_power_(kw),inverter_40_ac_power_(kw),meter_1_ac_power_kw_n_samples,meter_2_ac_power_kw_n_samples
utc_measured_on,,,,,,,,,,,,,,,,,,,,,
2016-02-17 13:30:00-05:00,42.727273,NaN,16.878182,8.277273,6.236000,NaN,5.470000,-0.878182,7.837273,10.430000,...,1.527273,1.654545,1.736364,1.554545,1.872727,1.690909,1.727273,1.563636,NaN,NaN
2016-02-17 14:30:00-05:00,-60.000000,NaN,13.508333,6.565833,1.270000,NaN,0.375000,-5.672727,5.912500,8.680000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,NaN,NaN
2016-02-17 15:30:00-05:00,-60.000000,NaN,13.336667,4.600000,1.270000,NaN,0.310000,-5.766667,4.455000,6.710833,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,NaN,NaN
2016-02-17 16:30:00-05:00,-50.000000,NaN,12.478333,3.510000,1.270000,NaN,-0.015000,-6.236364,2.930000,5.477500,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,NaN,NaN
2016-02-17 17:30:00-05:00,-59.166667,NaN,12.369091,2.249091,1.098333,NaN,-0.469583,-5.766667,1.706667,4.235833,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,NaN,NaN
